In [1]:
#Install Libraries
!pip install -q transformers datasets evaluate torchaudio accelerate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.1 MB/s eta 0:00:00


In [22]:
import transformers

print(transformers.__version__)

5.0.0


In [2]:
from datasets import load_from_disk, Audio
from transformers import AutoProcessor, AutoModelForCTC, TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nalaprogroup","key":"6e69ae1b1b69e1a4d23d6edf0680d84f"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d nalaprogroup/data-nalapro-project03

Dataset URL: https://www.kaggle.com/datasets/nalaprogroup/data-nalapro-project03
License(s): CC0-1.0
100% 354M/354M [00:02<00:00, 158MB/s]



In [6]:
import zipfile

zip_path = "data-nalapro-project03.zip"
extract_path = "/content/sample_data/data-nalapro-project03"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)

Extracted to: /content/sample_data/data-nalapro-project03


In [7]:
# Path dataset yang sudah kamu simpan
dataset_path = "/content/sample_data/data-nalapro-project03"
dataset_path

'/content/sample_data/data-nalapro-project03'

In [8]:
print("📥 Loading dataset...")
dataset = load_from_disk(dataset_path)
dataset

📥 Loading dataset...


DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

In [9]:
#Ambil hanya data train
train_dataset = dataset["train"]
val_dataset = dataset["validation"]

In [30]:
import IPython.display as ipd

sample = dataset["train"][0]

ipd.Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"])

In [10]:
train_dataset.column_names

['audio', 'intent_class', 'transcription']

In [11]:
# Hanya ambil kolom audio dan transcription
train_asr = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ['audio', 'transcription']]
)
val_asr = val_dataset.remove_columns(
    [col for col in val_dataset.column_names if col not in ['audio', 'transcription']]
)

In [12]:
# ============================================
# 4. RESAMPLE KE 16kHz (WAJIB!)
# ============================================
TARGET_SR = 16000
print(f"\n🎵 Resampling train data to {TARGET_SR} kHz...")
train_asr = train_asr.cast_column("audio", Audio(sampling_rate=TARGET_SR))
val_asr = val_asr.cast_column("audio", Audio(sampling_rate=TARGET_SR))


🎵 Resampling train data to 16000 kHz...


In [35]:
train_asr

Dataset({
    features: ['input_values', 'labels'],
    num_rows: 1800
})

In [13]:
# ============================================
# 5. NORMALISASI TEKS (UPPERCASE)
# ============================================
print("\n Normalizing transcriptions to uppercase...")

def uppercase_text(example):
    return {"transcription": example["transcription"].upper()}

train_asr = train_asr.map(uppercase_text)
val_asr = val_asr.map(uppercase_text)

# Cek sample
print(f"Sample transcription: {train_asr[0]['transcription'][:100]}...")

print("="*50)
print(f"Sample transcription: {val_asr[0]['transcription'][:100]}...")


 Normalizing transcriptions to uppercase...


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Sample transcription: BRAYON I RECENTLY AND I JUST WANT TO KNOW IF IT'S ALMOST OUT OF WORK I'M GOING NORTH AND GOING TO BE...
Sample transcription: I NEED INFORMATION ABOUT A BUSINESS PHONE...


In [14]:
print(train_asr.column_names)
print("="*50)
print(val_asr.column_names)

['audio', 'transcription']
['audio', 'transcription']


In [15]:
# ============================================
# 6. LOAD PROCESSOR
# ============================================
print("\nLoading Wav2Vec2 processor...")
processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
print(f"Processor loaded. Sampling rate: {processor.feature_extractor.sampling_rate} Hz")



Loading Wav2Vec2 processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

Processor loaded. Sampling rate: 16000 Hz


In [16]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    batch["labels"] = processor.tokenizer(
        batch["transcription"]
    ).input_ids

    return batch


print("\nApplying preprocessing to train data...")

train_asr = train_asr.map(
    prepare_dataset,
    remove_columns=train_asr.column_names,
    num_proc=1
)

print(f"Train data preprocessed: {len(train_asr)} samples")
print(f"Sample input_values length: {len(train_asr[0]['input_values'])}")
print(f"Sample labels length: {len(train_asr[0]['labels'])}")


print("\nApplying preprocessing to val data...")

val_asr = val_asr.map(
    prepare_dataset,
    remove_columns=val_asr.column_names,
    num_proc=1
)

print(f"Val data preprocessed: {len(val_asr)} samples")
print(f"Sample input_values length: {len(val_asr[0]['input_values'])}")
print(f"Sample labels length: {len(val_asr[0]['labels'])}")


Applying preprocessing to train data...


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Train data preprocessed: 1800 samples
Sample input_values length: 682666
Sample labels length: 175

Applying preprocessing to val data...


Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Val data preprocessed: 56 samples
Sample input_values length: 62806
Sample labels length: 41


In [17]:
print(train_asr.column_names)
print("="*50)
print(val_asr.column_names)

['input_values', 'labels']
['input_values', 'labels']


In [18]:
# ============================================
# 9. LOAD METRIK WER UNTUK EVALUASI
# ============================================
print("\n📊 Loading WER metric...")
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """
    Menghitung WER untuk evaluasi model
    """
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # Ganti -100 dengan pad_token_id untuk decoding
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions dan labels ke string
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    # Hitung WER
    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}



📊 Loading WER metric...


In [19]:
# ============================================
# 10. LOAD MODEL WAV2VEC2 BASE
# ============================================
print("\n🤖 Loading Wav2Vec2 Base model...")
model = AutoModelForCTC.from_pretrained(
    "facebook/wav2vec2-base",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

print(f"✅ Model loaded. Vocabulary size: {model.config.vocab_size}")
print(f"   Model parameters: {model.num_parameters():,}")


🤖 Loading Wav2Vec2 Base model...


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded. Vocabulary size: 32
   Model parameters: 94,396,320


In [20]:
# ============================================
# 11. TRAINING ARGUMENTS (DENGAN EVALUASI)
# ============================================
training_args = TrainingArguments(
    output_dir="./wav2vec2-minds14-asr",
    per_device_train_batch_size=4, # Reduced batch size to mitigate OOM error
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="steps",      # Evaluasi berkala
    eval_steps=200,                    # Evaluasi setiap 200 steps
    logging_steps=100,
    save_steps=200,                    # Fixed: Changed to be a multiple of eval_steps (200)
    num_train_epochs=3, # Reduced for faster testing after OOM
    learning_rate=3e-5,
    warmup_steps=500,
    fp16=True,
    gradient_checkpointing=True,
    save_total_limit=2,
    load_best_model_at_end=True,       # Load model terbaik berdasarkan eval loss
    metric_for_best_model="wer",       # Gunakan WER untuk menentukan model terbaik
    greater_is_better=False,           # WER lebih kecil lebih baik
    report_to="none",
)

In [23]:
print(transformers.__version__)

5.0.0


In [24]:
from dataclasses import dataclass
from typing import List, Dict, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels
        return batch

In [25]:
data_collator = DataCollatorCTCWithPadding(processor=processor)

In [26]:
# ============================================
# 12. INITIALIZE TRAINER (DENGAN VALIDATION)
# ============================================
print("\n🏋️ Initializing Trainer with validation set...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_asr,
    eval_dataset=val_asr,              # Validation dataset dari split awal
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


🏋️ Initializing Trainer with validation set...


In [27]:
# ============================================
# 13. START TRAINING
# ============================================
print("\n🚀 Starting training with validation monitoring...")
print("="*50)
trainer.train()


🚀 Starting training with validation monitoring...


Step,Training Loss,Validation Loss,Wer
200,20.286520,8.750231,0.979424
400,15.340504,7.067130,0.979424
600,13.029324,5.865180,0.994513


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=675, training_loss=23.797474410445602, metrics={'train_runtime': 1365.3226, 'train_samples_per_second': 3.955, 'train_steps_per_second': 0.494, 'total_flos': 7.659802511264333e+17, 'train_loss': 23.797474410445602, 'epoch': 3.0})